# 🚀 Optimized Video QA Pipeline - Google Colab Edition

**Two-stage optimized pipeline in a single notebook:**
- **Part 1**: Parallel frame extraction (CPU optimization)
- **Part 2**: Batch GPU inference (A100/V100 optimization)

**Features:**
- ⚡ Parallel processing (8 workers)
- 💾 Google Drive caching
- 🎯 Batch inference (size=4)
- 📊 Progress tracking
- 🔄 Reusable cached frames

### L4 GPU RUN FASTER THAN A100. Only cost 2.9 Compute per hours. with 22.5 VRAM (we don't need VRAM that much)

## 📁 Step 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/ZAIC-2025/traffic_buddy.zip" -d "/content"

Archive:  /content/drive/MyDrive/ZAIC-2025/traffic_buddy.zip
   creating: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/
  inflating: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/readme.md  
  inflating: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/.DS_Store  
   creating: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/
   creating: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos/
  inflating: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos/4efc8248_129_clip_007_0040_0045_N.mp4  
  inflating: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos/896490ab_442_clip_006_0040_0047_N.mp4  
  inflating: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos/8c99a894_100_clip_007_0037_0047_Y.mp4  
  inflating: /content/content/drive/MyDrive/ZA

## 📦 Step 2: Install Dependencies

In [ ]:
!lscpu # display cpu information

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      12
  On-line CPU(s) list:       0-11
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.20GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      6
    Socket(s):               1
    Stepping:                7
    BogoMIPS:                4400.42
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                 

In [ ]:
!pip install -q transformers qwen-vl-utils opencv-python pillow tqdm accelerate
!pip install -q flash-attn --no-build-isolation

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 100.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## ⚙️ Step 3: Configuration

In [ ]:
import os
from google.colab import userdata

# ============================================================================
# PATHS - Google Drive
# ============================================================================
# Update these paths based on your Google Drive structure
DATA_ROOT = "/content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test"
TRAIN_JSON = f"{DATA_ROOT}/train/train.json"

# Cache directory on Google Drive
CACHE_DIR = "/content/drive/MyDrive/zalo_ai_cache"
FRAMES_CACHE_DIR = os.path.join(CACHE_DIR, "extracted_frames")
RESULTS_DIR = os.path.join(CACHE_DIR, "results")

# Create directories
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(FRAMES_CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ============================================================================
# MODEL SETTINGS
# ============================================================================
# HF_TOKEN = userdata.get('HF_KEY')

# ============================================================================
# PREPROCESSING SETTINGS
# ============================================================================
NUM_WORKERS = 40  # Parallel workers for frame extraction

# ============================================================================
# INFERENCE SETTINGS
# ============================================================================
BATCH_SIZE = 4  # GPU batch size
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.1
DO_SAMPLE = False


print("✅ Configuration loaded")
print(f"📁 Data root: {DATA_ROOT}")
print(f"💾 Cache dir: {CACHE_DIR}")

✅ Configuration loaded
📁 Data root: /content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test
💾 Cache dir: /content/drive/MyDrive/zalo_ai_cache


## 🛠️ Step 4: Utility Functions

In [ ]:
%pip install opencv-python

In [ ]:
import json
import cv2
from PIL import Image

def load_json_data(json_path):
    """Load data from JSON file"""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    print(f"✅ Loaded {data['__count__']} questions")
    return data

def resolve_video_path(json_video_path):
    """Find absolute path of video"""
    abs_path = os.path.join(DATA_ROOT, json_video_path)
    if os.path.exists(abs_path):
        return abs_path
    raise FileNotFoundError(f"Video not found: {abs_path}")

def extract_frames_at_timestamps(video_path, support_frames):
    """Extract frames from video at specific timestamps"""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    extracted_frames = []
    frame_timestamps = []

    for timestamp in support_frames:
        center_frame = int(timestamp * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, center_frame)
        ret, frame = cap.read()

        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_image = Image.fromarray(frame_rgb)
            extracted_frames.append(pil_image)
            frame_timestamps.append(timestamp)

    cap.release()
    return extracted_frames, frame_timestamps



def parse_model_response(response):
    """Parse model response to extract observation, reasoning, and answer"""
    result = {'observation': '', 'reasoning': '', 'answer': ''}

    if 'QUAN SÁT:' in response:
        parts = response.split('QUAN SÁT:')
        if len(parts) > 1:
            obs_part = parts[1].split('SUY LUẬN:')[0]
            result['observation'] = obs_part.strip()

    if 'SUY LUẬN:' in response:
        parts = response.split('SUY LUẬN:')
        if len(parts) > 1:
            reason_part = parts[1].split('ĐÁP ÁN:')[0]
            result['reasoning'] = reason_part.strip()

    if 'ĐÁP ÁN:' in response:
        parts = response.split('ĐÁP ÁN:')
        if len(parts) > 1:
            answer_part = parts[1].strip()
            for char in answer_part:
                if char in ['A', 'B', 'C', 'D']:
                    result['answer'] = char
                    break

    return result

def extract_answer_letter(answer_text):
    """Extract answer letter (A, B, C, D) from text"""
    answer_text = answer_text.strip()
    if answer_text and answer_text[0] in ['A', 'B', 'C', 'D']:
        return answer_text[0]
    return None

def save_results_to_json(results, output_path):
    """Save results to JSON file"""
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved results to {output_path}")

print("✅ Utility functions loaded")

✅ Utility functions loaded


---
# PART 1: Preprocessing - Parallel Frame Extraction

Extract frames from all videos in parallel and cache to Google Drive

In [ ]:
import hashlib
import pickle
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

def get_cache_path(video_path, support_frames):
    """Generate unique cache path for extracted frames"""
    hash_input = f"{video_path}_{support_frames}".encode()
    cache_id = hashlib.md5(hash_input).hexdigest()
    cache_file = os.path.join(FRAMES_CACHE_DIR, f"{cache_id}.pkl")
    return cache_file

def extract_and_cache_frames(sample_data):
    """Extract frames for one sample and cache to disk"""
    try:
        video_path = resolve_video_path(sample_data['video_path'])
        cache_file = get_cache_path(video_path, sample_data['support_frames'])

        # Check if already cached
        if os.path.exists(cache_file):
            return {
                'id': sample_data['id'],
                'status': 'cached',
                'cache_path': cache_file
            }

        # Extract frames
        frames, timestamps = extract_frames_at_timestamps(
            video_path,
            sample_data['support_frames']
        )

        if not frames:
            return {
                'id': sample_data['id'],
                'status': 'failed',
                'error': 'No frames extracted'
            }

        # Save to cache
        with open(cache_file, 'wb') as f:
            pickle.dump({
                'frames': frames,
                'timestamps': timestamps,
                'id': sample_data['id'],
                'video_path': video_path,
                'support_frames': sample_data['support_frames']  # ← ADD THIS
            }, f)

        return {
            'id': sample_data['id'],
            'status': 'success',
            'cache_path': cache_file,
            'num_frames': len(frames)
        }

    except Exception as e:
        return {
            'id': sample_data['id'],
            'status': 'error',
            'error': str(e)
        }

def preprocess_dataset(data_list, num_workers=NUM_WORKERS):
    """Preprocess entire dataset in parallel"""
    print(f"\n🚀 Starting parallel preprocessing with {num_workers} workers")
    print(f"📁 Cache directory: {FRAMES_CACHE_DIR}")
    print(f"📊 Total samples: {len(data_list)}\n")

    results = []

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = {
            executor.submit(extract_and_cache_frames, sample): sample
            for sample in data_list
        }

        with tqdm(total=len(data_list), desc="Extracting frames") as pbar:
            for future in as_completed(futures):
                result = future.result()
                results.append(result)
                pbar.update(1)

                if result['status'] == 'error':
                    pbar.write(f"❌ Error {result['id']}: {result.get('error', 'Unknown')}")

    # Summary
    success_count = sum(1 for r in results if r['status'] in ['success', 'cached'])
    cached_count = sum(1 for r in results if r['status'] == 'cached')
    error_count = sum(1 for r in results if r['status'] == 'error')

    print(f"\n{'='*60}")
    print(f"📈 PREPROCESSING SUMMARY")
    print(f"{'='*60}")
    print(f"✅ Total successful: {success_count}/{len(data_list)}")
    print(f"💾 Already cached: {cached_count}")
    print(f"🆕 Newly processed: {success_count - cached_count}")
    print(f"❌ Errors: {error_count}")
    print(f"{'='*60}\n")

    return results

def create_index_file(results, output_path):
    """Create index file mapping question IDs to cache paths"""
    index = {
        r['id']: r['cache_path']
        for r in results
        if r['status'] in ['success', 'cached']
    }

    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(index, f, ensure_ascii=False, indent=2)

    print(f"✅ Created index file: {output_path}")
    return index

print("✅ Preprocessing functions loaded")

✅ Preprocessing functions loaded


### Run Preprocessing (Part 1)

This will extract frames from all videos in parallel.

**Note**: You can adjust the subset size for testing

---
# PART 2: Model Inference - Batch GPU Processing

Load cached frames and run batch inference on GPU

In [ ]:
# Load training data
train_data = load_json_data(TRAIN_JSON)

# For testing, use a subset (remove [:10] to process all)
# data_to_process = train_data['data'][:10]  # Test with 10 samples
data_to_process = train_data['data']  # Process all samples
# CACHE
# Run preprocessing
preprocessing_results = preprocess_dataset(data_to_process) # 40 workers for 1490 images ~ 1 min 41

# Create index file
index_path = os.path.join(CACHE_DIR, "frames_index.json")
frames_index = create_index_file(preprocessing_results, index_path)

print(f"\n✅ Preprocessing complete!")
print(f"📁 Frames cached in: {FRAMES_CACHE_DIR}")
print(f"📄 Index file: {index_path}")

✅ Loaded 1490 questions

🚀 Starting parallel preprocessing with 40 workers
📁 Cache directory: /content/drive/MyDrive/zalo_ai_cache/extracted_frames
📊 Total samples: 1490



Extracting frames: 100%|██████████| 1490/1490 [01:28<00:00, 16.86it/s]



📈 PREPROCESSING SUMMARY
✅ Total successful: 1301/1490
💾 Already cached: 1301
🆕 Newly processed: 0
❌ Errors: 0

✅ Created index file: /content/drive/MyDrive/zalo_ai_cache/frames_index.json

✅ Preprocessing complete!
📁 Frames cached in: /content/drive/MyDrive/zalo_ai_cache/extracted_frames
📄 Index file: /content/drive/MyDrive/zalo_ai_cache/frames_index.json


In [ ]:
MODEL_PATH = "Qwen/Qwen3-VL-4B-Instruct"
PROMPT_TEMPLATE = """
Bạn hãy đóng vai là **tài xế** của chiếc xe ô tô có gắn camera hành trình. Nhiệm vụ của bạn là phân tích tình huống giao thông từ chính góc nhìn của mình và trả lời câu hỏi.

LƯU Ý QUAN TRỌNG:
1.  Bạn chính là **chiếc xe ô tô** có gắn camera.
2.  Mọi câu hỏi (ví dụ: "xe đang chạy", "làn đường của xe", "ô tô") đều đang ám chỉ **bạn** và **làn đường của bạn**.
3.  Khi viết QUAN SÁT và SUY LUẬN, hãy dùng ngôi thứ nhất (ví dụ: "Tôi đang ở làn...", "Làn của tôi cho phép...").

Câu hỏi: {question}

Các lựa chọn:
{choices}

Hãy phân tích kỹ các khung hình và trả lời theo định dạng:

QUAN SÁT: [Mô tả chi tiết những gì bạn (tài xế) thấy từ góc nhìn của mình]
SUY LUẬN: [Giải thích cách bạn (tài xế) suy luận để đưa ra câu trả lời]
ĐÁP ÁN: [Chỉ ghi chữ cái: A, B, C, hoặc D]
"""


In [ ]:
def create_prompt(question, choices):
    """Create prompt for model"""
    choices_text = "\n".join(choices)
    return PROMPT_TEMPLATE.format(question=question, choices=choices_text)


🔧 Loading model and processor...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded on: cuda:0
✅ Dtype: torch.float16
Qwen3VLModel(
  (visual): Qwen3VLVisionModel(
    (patch_embed): Qwen3VLVisionPatchEmbed(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (pos_embed): Embedding(2304, 1024)
    (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-23): 24 x Qwen3VLVisionBlock(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): Qwen3VLVisionAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (mlp): Qwen3VLVisionMLP(
          (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (act_fn): GELUTanh()
        )
      )
    )
    (merger): Qwen3VLVisionPatchMerger

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from huggingface_hub import login

class CachedFramesDataset(Dataset):
    """Dataset that loads preprocessed frames from cache"""

    def __init__(self, data_list, frames_index):
        self.data_list = data_list
        self.frames_index = frames_index

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        sample = self.data_list[idx]

        # Load cached frames
        cache_path = self.frames_index.get(sample['id'])
        if cache_path and os.path.exists(cache_path):
            with open(cache_path, 'rb') as f:
                cached_data = pickle.load(f)
            frames = cached_data['frames']
        else:
            frames = []

        return {
            'id': sample['id'],
            'frames': frames,
            'question': sample['question'],
            'choices': sample['choices'],
            'answer': sample['answer'],
            'video_path': cached_data['video_path'],
            # 'support_frames': cached_data['support_frames'],
            'frame_timestamps': cached_data['timestamps']
        }


def load_model_and_processor():
    """Load Qwen model and processor"""
    print("\n🔧 Loading model and processor...")

    # Login to HuggingFace
    # if HF_TOKEN:
    #     login(HF_TOKEN)

    # Load model
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16, # Changed to float16 for 16-bit quantization
        attn_implementation="flash_attention_2",
        device_map="auto",
    )

    # Load processor
    processor = AutoProcessor.from_pretrained(MODEL_PATH)

    print(f"✅ Model loaded on: {model.device}")
    print(f"✅ Dtype: {model.dtype}")

    return model, processor

def inference_batch(model, processor, batch_samples):
    """Run inference on a batch of samples"""
    results = []

    for sample in batch_samples:
        try:
            frames = sample['frames']
            if not frames:
                results.append({
                    'id': sample['id'],
                    'error': 'No frames available'
                })
                continue

            # Create prompt
            print('SAMPLE:',sample)
            prompt = create_prompt(sample['question'], sample['choices'])

            # Prepare messages
            messages = [{"role": "user", "content": []}]
            for img in frames:
                messages[0]["content"].append({"type": "image", "image": img})
            messages[0]["content"].append({"type": "text", "text": prompt})

            # Process inputs
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(model.device)

            # Generate
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    temperature=TEMPERATURE,
                    do_sample=DO_SAMPLE
                )

            # Decode output
            generated_ids_trimmed = [
                out_ids[len(in_ids):]
                for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            )[0]

            # Parse response
            parsed = parse_model_response(output_text)
            ground_truth = extract_answer_letter(sample['answer'])
            is_correct = (parsed['answer'] == ground_truth)

            results.append({
                'id': sample['id'],
                'question': sample['question'],
                'predicted': parsed['answer'],
                'ground_truth': ground_truth,
                'correct': is_correct,
                'observation': parsed['observation'],
                'reasoning': parsed['reasoning'],
                'full_response': output_text,
                'video_path': sample['video_path'],
                'frame_timestamps': sample['frame_timestamps'],
                'support_frames': sample['support_frames']
            })

        except Exception as e:
            results.append({
                'id': sample['id'],
                'error': str(e)
            })

    return results

print("✅ Inference functions loaded")

✅ Inference functions loaded


### Load Model

In [ ]:
# Load model and processor
# model, processor = load_model_and_processor()

### Run Inference (Part 2)

Process all questions with batch inference

In [ ]:
BATCH_SIZE = 16
N = 16

# Load frames index
index_path = os.path.join(CACHE_DIR, "frames_index.json")
print(f"📄 Loading frames index: {index_path}")
with open(index_path, 'r', encoding='utf-8') as f:
    full_frames_index = json.load(f)
print(f"✅ Loaded {len(full_frames_index)} cached frame entries\n")

# Get only the first 8 items from frames_index
frames_index = dict(list(full_frames_index.items())[:N])
print(f"✅ Using {len(frames_index)} cached frame entries for subset\n")

# Create dataset and dataloader
# Select only the data samples whose IDs are in the selected frames_index
data_to_process_subset = [
    sample for sample in data_to_process if sample['id'] in frames_index
]

dataset = CachedFramesDataset(data_to_process_subset, frames_index)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=lambda x: x,  # Return batch as-is
    num_workers=0  # Must be 0 for GPU
)

print(f"🚀 Starting inference")
print(f"📊 Total samples: {len(data_to_process_subset)}")
print(f"📦 Batch size: {BATCH_SIZE}\n")

📄 Loading frames index: /content/drive/MyDrive/zalo_ai_cache/frames_index.json
✅ Loaded 1301 cached frame entries

✅ Using 16 cached frame entries for subset

🚀 Starting inference
📊 Total samples: 16
📦 Batch size: 16



In [ ]:
def load_model_and_processor():
    """Load Qwen model and processor"""
    print("\n🔧 Loading model and processor...")

    # Login to HuggingFace
    # if HF_TOKEN:
    #     login(HF_TOKEN)

    # Load model
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.float16, # Changed to float16 for 16-bit quantization
        attn_implementation="flash_attention_2",
        device_map="auto",
    )

    # Load processor
    processor = AutoProcessor.from_pretrained(MODEL_PATH)

    print(f"✅ Model loaded on: {model.device}")
    print(f"✅ Dtype: {model.dtype}")

    return model, processor


model, processor = load_model_and_processor()
print(model.model)


🔧 Loading model and processor...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model loaded on: cuda:0
✅ Dtype: torch.float16
Qwen3VLModel(
  (visual): Qwen3VLVisionModel(
    (patch_embed): Qwen3VLVisionPatchEmbed(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (pos_embed): Embedding(2304, 1024)
    (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-23): 24 x Qwen3VLVisionBlock(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): Qwen3VLVisionAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (mlp): Qwen3VLVisionMLP(
          (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (act_fn): GELUTanh()
        )
      )
    )
    (merger): Qwen3VLVisionPatchMerger

In [ ]:
all_results = []

# Process batches
with tqdm(total=len(data_to_process_subset), desc="Inference") as pbar:
    for batch in dataloader:
        batch_results = inference_batch(model, processor, batch)
        all_results.extend(batch_results)
        pbar.update(len(batch))

# Calculate accuracy
valid_results = [r for r in all_results if 'error' not in r]
correct_count = sum(1 for r in valid_results if r['correct'])
accuracy = (correct_count / len(valid_results) * 100) if valid_results else 0

print(f"\n{'='*60}")
print(f"📈 INFERENCE SUMMARY")
print(f"{'='*60}")
print(f"✅ Total processed: {len(valid_results)}/{len(data_to_process_subset)}")
print(f"🎯 Accuracy: {accuracy:.2f}%")
print(f"❌ Errors: {len(all_results) - len(valid_results)}")
print(f"{'='*60}\n")

Inference:   0%|          | 0/16 [00:00<?, ?it/s]

SAMPLE: {'id': 'train_0005', 'frames': [<PIL.Image.Image image mode=RGB size=2592x1944 at 0x7D56D015CC50>], 'question': 'Làn đường di chuyển hiện tại có được rẽ phải không?', 'choices': ['A. Có', 'B. Không'], 'answer': 'B. Không', 'video_path': '/content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos/2339ba19_386_clip_005_0032_0039_Y.mp4', 'frame_timestamps': [1.359896]}


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SAMPLE: {'id': 'train_0015', 'frames': [<PIL.Image.Image image mode=RGB size=2592x1944 at 0x7D56D015E270>], 'question': 'Trong video có xuất hiện biển báo nguy hiểm không?', 'choices': ['A. Có', 'B. Không'], 'answer': 'A. Có', 'video_path': '/content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos/b88139b9_144_clip_003_0018_0023_Y.mp4', 'frame_timestamps': [0.238415]}
SAMPLE: {'id': 'train_0017', 'frames': [<PIL.Image.Image image mode=RGB size=2592x1944 at 0x7D56D015FB30>], 'question': 'Trong video, biển báo giới hạn tốc độ áp dụng cho làn đường xe đang chạy đúng không?', 'choices': ['A. Đúng', 'B. Sai'], 'answer': 'A. Đúng', 'video_path': '/content/content/drive/MyDrive/ZAIC-2025/traffic_buddy_train+public_test/train/videos/5ebbcca5_144_clip_004_0023_0031_Y.mp4', 'frame_timestamps': [3.162689]}
SAMPLE: {'id': 'train_0018', 'frames': [<PIL.Image.Image image mode=RGB size=2592x1944 at 0x7D56D015EED0>], 'question': 'Tốc độ tối đa trên làn đường đang chạy là b

Inference: 100%|██████████| 16/16 [04:34<00:00, 17.19s/it]


📈 INFERENCE SUMMARY
✅ Total processed: 0/16
🎯 Accuracy: 0.00%
❌ Errors: 16



---
## 📊 Results Analysis

In [ ]:
incorrect_results = [r for r in valid_results if r['correct']]
incorrect_results

[]

In [ ]:
# Save results to Google Drive
output_path = os.path.join(RESULTS_DIR, "inference_results.json")
# save_results_to_json(all_results, output_path)

print(f"\n✅ Inference complete!")
print(f"📄 Results saved to: {output_path}")

# Analyze incorrect answers
incorrect_results = [r for r in valid_results if r['correct']]

print(f"\n{'='*60}")
print(f"❌ INCORRECT ANSWERS ({len(incorrect_results)} total)")
print(f"{'='*60}\n")

import textwrap

# Show first 5 incorrect answers
for i, result in enumerate(incorrect_results, 1):
    print(f"{i}. ID: {result['id']}")
    display_frames(result['id'], result['frame_timestamps'], result['video_path'], result['id'])
    print(f"   Question: {textwrap.fill(result['question'], width=80, subsequent_indent=' ')}")
    print(f"   Predicted: {result['predicted']}")
    print(f"   Ground Truth: {result['ground_truth']}")
    print(f"  👁️Observation: {textwrap.fill(result['observation'], width=80, subsequent_indent=' ')}")
    print(f"  🧠 Reasoning: {textwrap.fill(result['reasoning'], width=80, subsequent_indent=' ')}")
    print(f"   Video Path: {result['video_path'].split('/')[0]}")


✅ Inference complete!
📄 Results saved to: /content/drive/MyDrive/zalo_ai_cache/results/inference_results.json

❌ INCORRECT ANSWERS (0 total)



---
## 💡 Performance Tips

### For Preprocessing:
- Increase `NUM_WORKERS` if you have more CPU cores
- Cached frames are reused automatically
- Delete cache to force re-extraction

### For Inference:
- Adjust `BATCH_SIZE` based on GPU memory:
  - A100 40GB: batch_size=4-8
  - V100 16GB: batch_size=2-4
  - T4 16GB: batch_size=1-2
- Monitor GPU usage: `!nvidia-smi`

### Expected Performance:
- **Preprocessing**: ~2-5 minutes for 1490 videos (8 workers)
- **Inference**: ~15-30 minutes for 1490 questions (batch_size=4 on A100)
- **Total speedup**: 5-8x faster than sequential processing